# Overviews (image pyramids)

Overviews are down-sampled copies of a raster stored alongside the full-resolution data, so a
viewer can read a coarse level when zoomed out instead of the whole array — the *pyramid* the
package is named for.

- **`create_overviews`** — build the pyramid levels.
- **`overview_count`** — how many levels each band has.
- **`get_overview`** — fetch one level as a GDAL band.

## Setup

In [1]:
%matplotlib inline

import shutil
import tempfile
from pathlib import Path

import numpy as np

DATA = Path('../../../examples/data')
if not DATA.exists():
    DATA = Path('examples/data')
WORK = Path(tempfile.mkdtemp(prefix='pyramids-t2-'))
DATA.is_dir(), WORK.is_dir()

(True, True)

In [2]:
from pyramids.dataset import Dataset

# Copy to scratch first — create_overviews writes the pyramids into the file.
src = WORK / 'dem.tif'
shutil.copy(str(DATA / 'dem' / 'DEM5km_Rhine_burned_acc.tif'), str(src))
ds = Dataset.read_file(str(src))
ds.shape, ds.overview_count

2026-06-08 23:51:14 | INFO | pyramids.base.config | Logging is configured.


((1, 125, 93), [0])

## Build the pyramid — `create_overviews`

Pick a resampling method (`'average'`, `'nearest'`, `'cubic'`, …).

In [3]:
ds.create_overviews(resampling_method='average')
ds.overview_count  # levels per band

[7]

## Read a level — `get_overview`

Level 0 is the coarsest; each level halves the resolution.

In [4]:
ov = ds.get_overview(band=0, overview_index=0)
full = ds.shape
full, '->', (ov.XSize, ov.YSize)

((1, 125, 93), '->', (47, 63))

## Notes

- More levels = faster zoomed-out reads, slightly larger files.
- Cloud Optimized GeoTIFFs bake overviews into the file on write — see the COG tutorial.
- See also: [Visualization](visualization.ipynb) (the `plot(overview=True)` option).